# Lab 1: Infrastructure Provisining and AI Agents based application deployment

In this lab you will be Provision cloud infrastructure, deploy K3s, and run the Logistics Application with AI agents using Terraform and Ansible.

- Terraform-based infrastructure provisioning on IBM Cloud
- K3s installation and configuration
- Microservices deployment using Ansible
- Instana agent setup for observability

## Lab Introduction

### 📋 Lab Metadata

| Attribute | Details |
|-----------|---------|
| **Duration** | 60 minutes |
| **Skill Level** | Advanced |
| **Prerequisites** | IBM Cloud account, Terraform, Ansible, kubectl, Instana account |
| **Lab Type** | Hands-on, Step-by-step |
| **Execution Format** | Jupyter Notebook |


---

### 🎯 Learning Objectives

By the end of this lab, you will be able to:

1. ✅ Provision cloud infrastructure on IBM Cloud using Terraform
2. ✅ Deploy and configure K3s (lightweight Kubernetes) cluster
3. ✅ Deploy microservices applications using Ansible
5. ✅ Install and configure Instana agents for observability
6. ✅ Validate application deployment and functionality
---

### 🏗️ Architecture Overview

This lab deploys the following architecture:

<img src="./images/arch.png" alt="Architecture" width="800"/>


---
### 📦 Components to be Deployed
-  Infrastructure Components : VPC, VSI, SSH Key, Subnet, Security Group, Floating IP
- Kubernetes Components : K3s - Lightweight Kubernetes distribution
- Application Components : Front End, Auth, Order, Shipment, Tracking and AI Agents services and PostgreSQL database
- Observability Components : Instana Agent
---

### 📚 Prerequisites (Required Accounts)
- [ ] IBM Cloud account
- [ ] Instana account Access, Key, Host
- [ ] Watsonx.ai Project Id and Key
- [ ] Docker Credentials

---

## 1. Lab Execution - Provision VSI and K3s using Terraform

### 1.1 Preparation

#### 1.1.1 Generate SSH Key

1. In the second line below replace the **ga04** with your lab user id.

2. Generate SSH Key in your laptop by executing the below command

In [ ]:
import os

ssh_key_name = "t3-lab-key-ga04"

# Generate SSH key for VSI access
ssh_key_path = os.path.expanduser(f"~/.ssh/{ssh_key_name}")

# Check if SSH key already exists
if os.path.isfile(ssh_key_path) :
    print(f"✅ SSH key already exists at:\n{ssh_key_path}")
else:
    print("🔑 SSH key not found. Generating new SSH key...")

    !ssh-keygen -t rsa -b 4096 -f {ssh_key_path} -N "" -q

    print("\nSSH key :")
    print(f"{ssh_key_path}")

print("\nPublic key:")
print(f"{ssh_key_path}.pub")

# Display public key
print("\nPublic key content:")
!cat {ssh_key_path}.pub

2. Copy the content of **Public key content** to a notepad. (ssh-rsa AAAAB3NzaC1yc2E...)

#### 1.1.2 Get Other keys from Lab Instrctor.

Hope the instructor already shared you the following. If not get it from him.

- IBM Cloud API Key
- Lab User Id
- Instana Key
- Instana Host
- Watsonx Project Id
- Watsonx Key
- Docker Secret

#### 1.1.3 Find and Replace keys and other values

1. Replace each **Search Text** with its corresponding value (as per the **Value** column) in all files within the **files** folder.

| Search Text | Value | Sample Values |
|-----------|---------|---------|
| ##LAB_USER_ID## | Unique Id received from the lab instructor. It is good to have 4 characters ex: ga01 or sg01, sg02, etc. It should be a to z and 0 to 9| sg01, sg02, sg03 |
|    |  |.  |
| ##IBM_CLOUD_API_KEY## | IBM Cloud API Key received form the lab instructor  ||
| ##DOCKER_SECRET_CONFIG## | Docker Secret Config received from Instructor| eyJhdXRocyI6e ........ hadiJ9fX0= | 
|    |  |.  |
| ##INSTANA_KEY## | Instana Key received form the lab instructor ||
| ##INSTANA_HOST## | Instana Host received form the lab instructor |ingress-orange-saas.instana.io|
|    |  |.  |
| ##WATSONX_PROJECT_ID## | Watsonx Project Id received form the lab instructor ||
| ##WATSONX_KEY## | Watsonx key received from the lab instructor ||
|    |  |.  |
| ##SSH_PUBLIC_KEY## | SSH Key that you have retrived in the previous step |ssh-rsa AAAAB3......o.local|
| ##SSH_KEY_NAME## | The text "t3-lab-ssh-key-" suffixed with your lab user id| t3-lab-ssh-key-sg01 ,t3-lab-ssh-key-sg02 |
|    |  |.  |
| ##JWT_SECRET_KEY## | Any secret key as per your wish | your-secret-key-min-32-characters-long-for-security |
| ##POSTGRES_PASSWORD## | Your own password for postgres |mypamypamypa2000|

#### Afer the replacement some values are like this.

Lets say the Unique Id received from instructor is ga01, the values could like the below. (Not all the variables are given here)

| Variables | Values |
|-----------|---------|
| vsi_name | t3-lab-vsi-sg01 |
| k3s cluster name | t3-lab-k3s-cluster-sg01 |
| Instana Key | xxxxxxx |
| Instana Host | ingress-orange-saas.instana.io |
| Instana Zone Name  | t3-lab-k3s-zone-sg01 |
| ibmcloud_api_key | xxx-xxxxxx_xxxx |
| ssh_public_key | ssh-rsa AAAAB3......o.local |

### 1.2 Execution

#### 1.2.1 Create SSH with Terraform

SSH Key name should be unique in the IBM Cloud Account. It is good to suffix with lab user id to avoid duplicates. (Ex: t3-lab-ssh-key-ga01 , t3-lab-ssh-key-ga02)

1. Execute the below 3 steps.

In [ ]:
%%bash
# Initialize Terraform
cd files/terraform/ssh
terraform init

In [ ]:
%%bash
# Review the execution plan
cd files/terraform/ssh
terraform plan

While executing the below, if you get an error "Error: [DEBUG] Create SSH Key Key with name already exists" then you can ingore it.

In [ ]:
%%bash
# Apply the configuration
cd files/terraform/ssh
terraform apply -auto-approve


#### 1.2.2 Create VSI and K3s with Terraform

1. Execute the below 3 steps. Approximately it would take 5 to 10 minutes.

In [ ]:
%%bash
# Initialize Terraform
cd files/terraform/vsi
terraform init

In [ ]:
%%bash
# Review the execution plan
cd files/terraform/vsi
terraform plan

In [ ]:
%%bash
# Apply the configuration
cd files/terraform/vsi
terraform apply -auto-approve

Please wait 2–3 minutes for provisioning to complete.


#### 1.2.2 Resources Created

This configuration automatically creates:

1. **VPC**: Virtual Private Cloud for network isolation
2. **Subnet**: Network subnet within the VPC (256 IP addresses)
3. **SSH Key**: Uploaded to IBM Cloud for secure access
4. **Security Group**: With rules for SSH (22), HTTP (80), HTTPS (443), and K3s API (6443)
5. **Virtual Server Instance (VSI)**: Ubuntu 22.04 with 2 vCPUs and 8GB RAM
6. **Floating IP**: Public IP address for external access
7. **K3s Cluster**: Fully functional single-node Kubernetes cluster

---

### 1.3 Verification

#### 1.3.1 Get the Floating IP of the VSI

1. Execute the below to get the floating IP of the VSI.

2. Make a note of this FLOATING_IP, this will be used in several places in this lab.

In [ ]:
%%bash
# Get the floating IP from Terraform output
cd files/terraform/vsi
FLOATING_IP=$(terraform output -raw vsi_floating_ip)
echo "Floating IP is : $FLOATING_IP"

#### 1.3.2 Find and Replace FLOATING_IP

1. Find and replace all the occurrences of **##FLOATING_IP##** with the obtained Floating IP in all the files under the **files** folder.

#### 1.3.3 SSH into the VSI

1. Open a Terminal Window or Command line Window

2. Run the below command after replacing the <LAB_USER_ID> and <FLOATING_IP>

```bash
ssh -i ~/.ssh/t3-lab-key-<LAB_USER_ID> ubuntu@<FLOATING_IP>
```

#### 1.3.4 Accessing K3s cluster from VSI

in the above terminal window, run the below commands to access K3s cluster.

```bash
# Check K3s status
sudo systemctl status k3s

# View cluster nodes
sudo kubectl get nodes

# View all pods
sudo kubectl get pods -A
```

---

## 2. Lab Execution - Deploy Application using Ansible

### 2.1 Preparation

Nil

---

### 2.2 Execution

#### 2.2.1 Deploy Application using Ansible

1. Execute the below.

The deployment takes approximately 2-3 minutes.

In [ ]:
%%bash
# Run the deployment playbook
cd files/ansible
ansible-playbook deploy-microservice.yml

#### 2.2.2 Create Tables and Records in DB

1. Replace the ssh_key_name with your key name
2. Replace the FloatingIP with your floating IP value
3. Execute the commands below

In [ ]:
%%bash
cd files/ansible/db-data
ssh_key_name=t3-lab-key-ga04
FloatingIP=111.222.333.444
ssh -i ~/.ssh/$ssh_key_name ubuntu@$FloatingIP "sudo kubectl exec -it -n logistics deployment/postgres -- psql -U logistics_user -d logistics_db" < database-schema.sql
ssh -i ~/.ssh/$ssh_key_name ubuntu@$FloatingIP "sudo kubectl exec -it -n logistics deployment/postgres -- psql -U logistics_user -d logistics_db" < database-data.sql

#### 2.2.3 What Gets Deployed

- Namespace : logistics
- Services : frontend, auth, order, shipment, tracking, postgresql
- Others: api-gateway
---

### 2.3 Verification

#### 2.3.1 SSH into the VSI

You can skip this section if your SSH window is still open and active.

1. Open a Terminal Window or Command line Window

2. Run the below command after replacing the <LAB_USER_ID> and <FLOATING_IP>

```bash
ssh -i ~/.ssh/t3-lab-key-<LAB_USER_ID> ubuntu@<FLOATING_IP>
```

#### 2.3.2 Accessing application from K3s cluster deployud in VSI

1. In the above terminal window, run the below commands to access K3s cluster.

```bash
# Check pods
sudo kubectl get pods -n logistics

# Check services
sudo kubectl get svc -n logistics
```

You may get the output like this. 

2. For  **api-gateway** service, make a note of **NodePort IP** (111.222.333.444) and **Port** 32283

ubuntu@t3-lab-vsi-ga03:/tmp$ sudo kubectl get svc -n logistics

|NAME                |TYPE        |CLUSTER-IP      |EXTERNAL-IP   |PORT(S)        |AGE|
|-----------|---------|---------|---------|---------|---------|
|api-gateway        | NodePort    |111.222.333.444| <none>        |80:32283/TCP   |16h|
|auth-service       | ClusterIP   |10.43.58.13|     <none>        |8001/TCP       |16h|
|frontend-service   | ClusterIP   |10.43.189.62|    <none>        |8000/TCP       |16h|
|order-service       |ClusterIP   |10.43.233.184|   <none>        |8002/TCP       |16h|
|postgres-headless   |ClusterIP   |None|            <none>        |5432/TCP       |12h|
|postgres-service    |ClusterIP   |10.43.250.178|   <none>        |5432/TCP       |12h|
|shipment-service    |ClusterIP   |10.43.181.208|   <none>        |8003/TCP       |16h|
|tracking-service    |ClusterIP   |10.43.56.50|     <none>        |8004/TCP       |16h|

#### 2.3.3 Test the APIs

1. Open the below URLs in the browser (after replacing the NodePortIP and Port with the above values)

http://NodePortIP:Port/auth/

http://NodePortIP:Port/order/

http://NodePortIP:Port/shipmnent/

http://NodePortIP:Port/tracking/

#### 2.3.4 Test the application

1. Open the below url in the browser after replacing the < NodePortIP > and < Port > with the above values.

http://NodePortIP:Port

---

## 3. Lab Execution - Deploy Instana Agent using Ansible

For comprehensive monitoring and observability of your K3s cluster, you can install the Instana remote agent.

### 3.1 Preparation

Nil

### 3.2 Execution

#### 3.2.1 Deploy Instana Agent using Ansible

In [ ]:
%%bash
# Run the Instana deployment playbook
cd files/ansible
ansible-playbook deploy-instana-agent.yml

#### 3.2.2 What Gets Installed

- **Instana Agent Operator** - Manages agent lifecycle
- **Instana Remote Agent** - Collects monitoring data (DaemonSet)
- **Custom Resources** - InstanaAgentRemote CRD

### 3.3 Verification


#### 3.3.1 SSH into the VSI

You can skip this section if your SSH window is still open and active.

1. Open a Terminal Window or Command line Window

2. Run the below command after replacing the <LAB_USER_ID> and <FLOATING_IP>

```bash
ssh -i ~/.ssh/t3-lab-key-<LAB_USER_ID> ubuntu@<FLOATING_IP>
```


#### 3.3.2 Accessing instana-agent from K3s cluster deployud in VSI

in the above terminal window, run the below commands to access K3s cluster.

```bash
# Check Instana resources
sudo kubectl get all -n instana-agent

# Check agent logs
sudo kubectl logs -f -n instana-agent -l app.kubernetes.io/name=instana-agent
```


#### 3.3.3 View the Cluster in Instana

1. Open the Instana Console in the browser
2. Goto **Platforms > Kubernetes** page
3. Search for **t3-lab**. you will find the your cluster listed there.